# Regular expressions: follow along

This notebook accompanies the slides. Run it from `notebooks/` with `../data/talks_sample.csv` available. Predict each result before running the next cell. The slides also contain the outputs.

In [1]:
import re
from pathlib import Path
import pandas as pd

print("Ready to explore regex")

Ready to explore regex


## Recall last week's work

If income is missing more often for older respondents, observed age predicts missingness. That is evidence against MCAR, but does not establish MAR over MNAR. When a cleaning rule changes, preserve raw data and record which pipeline version produced each output.

## Search or fullmatch?

Predict what happens if the five-digit pattern looks inside a labeled string versus checking the entire field.

In [2]:
print("Search:", re.search(r"\d{5}", "ZIP: 84043").group())
print("Whole labeled field:", re.fullmatch(r"\d{5}", "ZIP: 84043"))
print("Whole ZIP field:", re.fullmatch(r"\d{5}", "84043").group())

Search: 84043
Whole labeled field: None
Whole ZIP field: 84043


## ZIP codes

`\d{5}` means five digits. An optional `-` followed by four digits admits ZIP+4. A word boundary prevents the pattern from beginning in the middle of a word.

In [3]:
zip_pattern = r"\b\d{5}(?:-\d{4})?\b"
examples = ["84043", "84043-4820", "ID84043X", "8404"]
for value in examples:
    hit = re.search(zip_pattern, value)
    print(f"{value:12} -> {hit.group() if hit else None}")
print("Whole-field check:", bool(re.fullmatch(zip_pattern, "ZIP: 84043")))

84043        -> 84043
84043-4820   -> 84043-4820
ID84043X     -> None
8404         -> None
Whole-field check: False


## Four whiteboard patterns

Pause here. Groups A–D first write a **short pattern only** for their assigned whole-field problem. Explain the rejected example. After the board discussion, run the four patterns below and compare their results.

In [4]:
board_patterns = {
    "A ZIP": (r"\d{5}(?:-\d{4})?", ["84043", "84043-4820", "84043-482"]),
    "B phone": (r"\d{3}[-.]\d{3}[-.]\d{4}", ["801-555-0142", "801.555.0142", "8015550142"]),
    "C email": (r"[A-Za-z0-9]+@byu\.edu", ["tom@byu.edu", "tom7@byu.edu", "tom@byuXedu"]),
    "D citation": (r"(?:John|Alma) \d+:\d+", ["John 8:31", "Alma 34:32", "Mosiah 3:19"]),
}
for group, (pattern, cases) in board_patterns.items():
    print(group, [bool(re.fullmatch(pattern, value)) for value in cases])

A ZIP [True, True, False]
B phone [True, True, False]
C email [True, True, False]
D citation [True, True, False]


## Phone numbers

The area code can be followed by punctuation, or can be surrounded by literal parentheses. `(?:A|B)` groups alternatives without capturing an output column.

In [5]:
phone_pattern = r"(?:\(\d{3}\)\s*|\d{3}[-. ])\d{3}[-. ]\d{4}"
numbers = ["801-555-0142", "(801) 555-0142", "801.555.0142", "8015550142"]
for value in numbers:
    print(f"{value:16} -> {bool(re.fullmatch(phone_pattern, value))}")

801-555-0142     -> True
(801) 555-0142   -> True
801.555.0142     -> True
8015550142       -> False


## Email addresses

Here we are extracting common email shapes from unstructured text. This short pattern is not intended to validate all legitimate email addresses.

In [6]:
email_pattern = r"[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}"
line = "Write first.last+class@byu.edu or help@example.org."
print(re.findall(email_pattern, line))

['first.last+class@byu.edu', 'help@example.org']


## Two real talks

These are a small sample from the course dataset. Inspect the original strings before writing a rule. In particular, the space between a number and `Nephi` is not always an ordinary ASCII space.

In [7]:
talks = pd.read_csv(Path("../data/talks_sample.csv"))
print(talks[["talk_name", "speaker", "conference"]].to_string(index=False))
first = re.search(r"1\s+Nephi\s+\d+:\d+", talks.loc[0, "text"])
print("Raw citation:", repr(first.group()))

                            talk_name        speaker   conference
                      Now Is the Time Jack N. Gerard October 2018
The Sacrament Can Help Us Become Holy Peter F. Meurs October 2016
Raw citation: '1\xa0Nephi 8:27'


## Begin with a single book

What information does this pattern lose from `John 8:31–32`?

In [8]:
print(re.findall(r"John\s+\d+:\d+", talks.loc[0, "text"]))

['John 8:31', 'John 14:17', 'John 16:33']


## Capture the parts

The group names become column names. `\s+` also matches a nonbreaking space. `[-–]` accepts the two range separators we want to handle today.

In [9]:
citation_pattern = (
    r"\b(?P<book>Doctrine\s+and\s+Covenants|D&C|[1-3]\s+Nephi|John|Alma|Mosiah)"
    r"\s+(?P<chapter>\d{1,3}):(?P<verse_start>\d{1,3})"
    r"(?:[-–](?P<verse_end>\d{1,3}))?\b"
)
print("One reference:", re.search(citation_pattern, "Alma 34:32–33").groupdict())
matches = talks["text"].str.extractall(citation_pattern)
citations = matches.reset_index().rename(
    columns={"level_0": "talk_row", "match": "citation_number"}
)
citations.insert(0, "talk_name", talks.loc[citations["talk_row"], "talk_name"].to_numpy())
print(citations[["talk_name", "book", "chapter", "verse_start", "verse_end"]].to_string(index=False))

One reference: {'book': 'Alma', 'chapter': '34', 'verse_start': '32', 'verse_end': '33'}
                            talk_name                   book chapter verse_start verse_end
                      Now Is the Time                1 Nephi       8          27       NaN
                      Now Is the Time                1 Nephi       8          28       NaN
                      Now Is the Time Doctrine and Covenants     121          35       NaN
                      Now Is the Time                   Alma      34          32        33
                      Now Is the Time                   John       8          31        32
                      Now Is the Time                   John      14          17       NaN
                      Now Is the Time                   John      16          33       NaN
The Sacrament Can Help Us Become Holy                3 Nephi      18          11       NaN
The Sacrament Can Help Us Become Holy                 Mosiah       3          19       NaN
T

## Three pandas operations

`.str.contains()` answers whether a talk contains a chosen pattern. `.str.extract()` returns only the first captured match. `.str.extractall()` supplies one row per match, which is what we need when a talk mentions several references.

In [10]:
has_john = talks["text"].str.contains(r"\bJohn\s+\d+:\d+", regex=True)
first_john = talks["text"].str.extract(r"\b(John\s+\d+:\d+)")[0]
print(pd.DataFrame({"talk_name": talks["talk_name"], "has_john": has_john, "first_john": first_john}).to_string(index=False))

                            talk_name  has_john first_john
                      Now Is the Time      True  John 8:31
The Sacrament Can Help Us Become Holy     False        NaN


## Normalize without deleting the source

The raw `text` and the extracted `book` fields remain in the data. The standardized book name goes into a separate field.

In [11]:
citations["book_clean"] = (
    citations["book"]
    .str.replace("\u00a0", " ", regex=False)
    .replace({"D&C": "Doctrine and Covenants"})
)
print(citations.groupby("talk_name").size().to_string())
print(citations.loc[citations["book"] == "D&C", ["book", "book_clean"]].to_string(index=False))

talk_name
Now Is the Time                          7
The Sacrament Can Help Us Become Holy    6
book             book_clean
 D&C Doctrine and Covenants


## Greedy matching

Two sets of parentheses give `.*` a chance to match far more than one reference.

In [12]:
sentence = "(John 8:31) and (Alma 7:12)"
print("Greedy:", re.findall(r"\(.*\)", sentence))
print("Lazy:  ", re.findall(r"\(.*?\)", sentence))

Greedy: ['(John 8:31) and (Alma 7:12)']
Lazy:   ['(John 8:31)', '(Alma 7:12)']


## Check what the pattern misses

The observed counts are counts produced by this particular pattern. A good next step is to inspect uncaptured references and expand the book list in the independent practice notebook.

In [13]:
expected = {"Now Is the Time": 7, "The Sacrament Can Help Us Become Holy": 6}
observed = citations.groupby("talk_name").size().to_dict()
assert observed == expected, (observed, expected)
print("Observed counts match the inspected examples:", observed)

Observed counts match the inspected examples: {'Now Is the Time': 7, 'The Sacrament Can Help Us Become Holy': 6}
